In [14]:
import numpy as np
from qdk_chemistry.data import Element, Structure, MajoranaMapping
from qdk_chemistry.algorithms import create
from qdk_chemistry.utils import Logger

Logger.set_global_level(Logger.LogLevel.off)

coords_ang = np.array(
    [
        [0.0, 0.0, 0.0],
        [0.0, 0.0, 0.740848],
    ]
)
structure = Structure(coords_ang, [Element.H, Element.H])

scf = create("scf_solver")
scf_energy, wavefunction = scf.run(
    structure, charge=0, spin_multiplicity=1, basis_or_guess="sto-3g"
)

# 2) Electronic Hamiltonian
ham_constructor = create("hamiltonian_constructor")
hamiltonian = ham_constructor.run(wavefunction.get_orbitals())

# 3) Qubit Hamiltonian
mapper = create("qubit_mapper")
n_spin_orbitals = hamiltonian.get_orbitals().get_num_molecular_orbitals() * 2
qubit_hamiltonian = mapper.run(hamiltonian, MajoranaMapping.jordan_wigner(n_spin_orbitals))

# 4) Ground-state energy of the qubit Hamiltonian
solver = create("qubit_hamiltonian_solver")
qubit_energy, ground_state = solver.run(qubit_hamiltonian)
qubit_energy = qubit_energy

In [ ]:
import scipy
import math
from qdk_chemistry.algorithms.hamiltonian_unitary_builder.time_evolution.trotter import Trotter
from qdk_chemistry.algorithms.hamiltonian_unitary_builder.time_evolution.trotter_error import trotter_steps_commutator
import numpy as np
import pandas as pd

_PAULI = {
    "I": np.eye(2, dtype=complex),
    "X": np.array([[0, 1], [1, 0]], dtype=complex),
    "Y": np.array([[0, -1j], [1j, 0]], dtype=complex),
    "Z": np.array([[1, 0], [0, -1]], dtype=complex),
}

def _pauli_matrix(pauli_map: dict[int, str], num_qubits: int) -> np.ndarray:
    """Build the full 2^n x 2^n matrix for a sparse Pauli map {qubit_idx: 'X'/'Y'/'Z'}."""
    mat = np.array([[1.0]], dtype=complex)
    for q in range(num_qubits - 1, -1, -1):
        mat = np.kron(mat, _PAULI[pauli_map.get(q, "I")])
    return mat

def numerical_error_sweep(hamiltonian, t=0.1, num_divisions=1, order=1):
    num_qubits = hamiltonian.num_qubits
    builder = Trotter(num_divisions=num_divisions, time=t, order=order)
    unitary = builder.run(hamiltonian)
    container = unitary.get_container()

    # Build Trotter unitary matrix
    dim = 2**num_qubits
    u_step = np.eye(dim, dtype=complex)
    for term in container.step_terms:
        pauli_mat = _pauli_matrix(term.pauli_term, num_qubits)
        u_step = scipy.linalg.expm(-1j * term.angle * pauli_mat) @ u_step
    u_trot = np.linalg.matrix_power(u_step, container.step_reps)

    # Exact unitary
    hamiltonian_matrix = hamiltonian.to_matrix()
    u_exact = scipy.linalg.expm(-1j * t * hamiltonian_matrix)

    # State-specific error: ||U_trot|psi0> - U_exact|psi0>||_2
    psi_trot = u_trot @ ground_state
    psi_exact = u_exact @ ground_state
    ground_state_error = np.linalg.norm(psi_trot - psi_exact)

    # # Operator error: ||U_trot - U_exact||_2
    # operator_error = np.linalg.norm(u_trot - u_exact)

    return ground_state_error

t = math.pi/qubit_hamiltonian.schatten_norm
target_accuracy = 1e-2
sweep_results = []
theoretical_params = []
optimal_params = []
for order in [1, 2, 4]:
    for num_phase_bit in range(6, 11, 2):
        scaled_t = t * 2**(num_phase_bit - 1)
        step = trotter_steps_commutator(qubit_hamiltonian, scaled_t, target_accuracy, order=order)
        theoretical_params .append({
            "order": order,
            "num_phase_bit": num_phase_bit,
            "t": t,
            "scaled_t": scaled_t,
            "num_divisions": step,
        })
        print(f"Theoretical parameters for order={order}, num_phase_bit={num_phase_bit}: num_divisions={step}")
        for num_divisions in range(1, 5000, 20):
            ground_state_error = numerical_error_sweep(qubit_hamiltonian, t=scaled_t, num_divisions=num_divisions, order=order)
            sweep_results.append({
                "order": order,
                "num_phase_bit": num_phase_bit,
                "t": t,
                "scaled_t": scaled_t,
                "num_divisions": num_divisions,
                # "operator_error": operator_error,
                "ground_state_error": ground_state_error,
            })
            if ground_state_error < target_accuracy:
                optimal_params.append({
                    "order": order,
                    "num_phase_bit": num_phase_bit,
                    "t": t,
                    "scaled_t": scaled_t,
                    "num_divisions": num_divisions,
                    "ground_state_error": ground_state_error,
                })
                print(f"Found optimal parameters for order={order}, num_phase_bit={num_phase_bit}: num_divisions={num_divisions}, ground_state_error={ground_state_error:.3e}")
                break

result = pd.DataFrame(sweep_results)
result


Found optimal parameters for order=1, num_phase_bit=6: num_divisions=221, theoretical_num_divisions=16708, ground_state_error=9.523e-03
Found optimal parameters for order=1, num_phase_bit=8: num_divisions=1141, theoretical_num_divisions=16708, ground_state_error=9.965e-03
Found optimal parameters for order=2, num_phase_bit=6: num_divisions=101, theoretical_num_divisions=16708, ground_state_error=8.758e-03
Found optimal parameters for order=2, num_phase_bit=8: num_divisions=761, theoretical_num_divisions=16708, ground_state_error=9.774e-03
Found optimal parameters for order=4, num_phase_bit=6: num_divisions=21, theoretical_num_divisions=16708, ground_state_error=6.455e-03
Found optimal parameters for order=4, num_phase_bit=8: num_divisions=101, theoretical_num_divisions=16708, ground_state_error=7.401e-03
Found optimal parameters for order=4, num_phase_bit=10: num_divisions=541, theoretical_num_divisions=16708, ground_state_error=9.514e-03


,order,num_phase_bit,t,scaled_t,num_divisions,ground_state_error
0,1,6,0.995936,31.869963,1,1.458720
1,1,6,0.995936,31.869963,21,0.690729
2,1,6,0.995936,31.869963,41,0.141654
3,1,6,0.995936,31.869963,61,0.065543
4,1,6,0.995936,31.869963,81,0.039778
...,...,...,...,...,...,...
646,4,10,0.995936,509.919404,461,0.017864
647,4,10,0.995936,509.919404,481,0.015132
648,4,10,0.995936,509.919404,501,0.012893
649,4,10,0.995936,509.919404,521,0.011046


In [16]:
theoretical_params_df = pd.DataFrame(theoretical_params)
theoretical_params_df.to_csv("theoretical_params.csv", index=False)
theoretical_params_df

""


In [17]:
optimal_params_df = pd.DataFrame(optimal_params)
optimal_params_df.to_csv("optimal_params.csv", index=False)
optimal_params_df

,order,num_phase_bit,t,scaled_t,num_divisions,ground_state_error
0,1,6,0.995936,31.869963,221,0.009523
1,1,8,0.995936,127.479851,1141,0.009965
2,2,6,0.995936,31.869963,101,0.008758
3,2,8,0.995936,127.479851,761,0.009774
4,4,6,0.995936,31.869963,21,0.006455
5,4,8,0.995936,127.479851,101,0.007401
6,4,10,0.995936,509.919404,541,0.009514


In [ ]:
from utils.qpe_utils import compute_evolution_time
from qdk_chemistry.data import AlgorithmRef
from qdk_chemistry.algorithms.state_preparation import identity_state_prep
import pandas as pd


def run_trotter(m_precision, trotter_order, num_divisions, evolution_time):
    unitary_builder = AlgorithmRef("hamiltonian_unitary_builder", "trotter", time=evolution_time, order=trotter_order, power_strategy="rescale", num_divisions=num_divisions)
    circuit_mapper = AlgorithmRef("controlled_circuit_mapper", "pauli_sequence")
    iqpe_circuit_builder = create(
        "qpe_circuit_builder", 
        "qdk_iterative",
        num_bits=m_precision,
        unitary_builder=unitary_builder,
        controlled_circuit_mapper=circuit_mapper,
    )
    state_prep = identity_state_prep(num_qubits=qubit_hamiltonian.num_qubits)
    circuits = iqpe_circuit_builder.run(state_prep, qubit_hamiltonian)
    largest_circuit = circuits[0]
    result = largest_circuit.estimate()
    logical_estimate = result["logicalCounts"]

    return logical_estimate

estimate_results = []
for row in optimal_params:
    m_precision = row["num_phase_bit"]
    trotter_order = row["order"]
    num_divisions = row["num_divisions"]
    evolution_time = row["scaled_t"]
    logical_estimate = run_trotter(m_precision, trotter_order, num_divisions, evolution_time)
    estimate_results.append({
        "type": "optimal",
        "m_precision": m_precision,
        "trotter_order": trotter_order,
        "num_divisions": num_divisions,
        "evolution_time": evolution_time,
        "numQubits": logical_estimate["numQubits"],
        "rotationCount": logical_estimate["rotationCount"],
        "rotationDepth": logical_estimate["rotationDepth"],
    })

for row in theoretical_params:
    m_precision = row["num_phase_bit"]
    trotter_order = row["order"]
    num_divisions = row["num_divisions"]
    evolution_time = row["scaled_t"]
    logical_estimate = run_trotter(m_precision, trotter_order, num_divisions, evolution_time)
    estimate_results.append({
        "type": "theoretical",
        "m_precision": m_precision,
        "trotter_order": trotter_order,
        "num_divisions": num_divisions,
        "evolution_time": evolution_time,
        "numQubits": logical_estimate["numQubits"],
        "rotationCount": logical_estimate["rotationCount"],
        "rotationDepth": logical_estimate["rotationDepth"],
    })
df = pd.DataFrame(estimate_results)
df


,m_precision,trotter_order,num_divisions,evolution_time,numQubits,rotationCount,rotationDepth
0,6,1,221,31.869963,5,6409,5525
1,8,1,1141,127.479851,5,33089,28525
2,6,2,101,31.869963,5,5656,4848
3,8,2,761,127.479851,5,42616,36528
4,6,4,21,31.869963,5,5712,4872
5,8,4,101,127.479851,5,27472,23432
6,10,4,541,509.919404,5,147152,125512


In [21]:
logical_estimate

{'numQubits': 5,
 'tCount': 0,
 'rotationCount': 147152,
 'rotationDepth': 125512,
 'cczCount': 0,
 'ccixCount': 0,
 'measurementCount': 1}